<a href="https://colab.research.google.com/github/hemanthgm914856/CANARA_BANK_CHATBOT/blob/main/CANARA_BANK_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface sentence-transformers pypdf beautifulsoup4 faiss-cpu openai

from openai import OpenAI
from google.colab import userdata

from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Step 1: Ingesting Canara Bank data...")

raw_paragraphs = [
    "Canara Bank is one of India's major public sector banks. It was founded in 1906 and its headquarters is located in Bengaluru, Karnataka.",
    "Canara Bank provides banking services such as savings accounts, current accounts, deposits, loans, credit cards, debit cards, internet banking and mobile banking.",
    "Canara Bank offers various loan products including education loans, home loans, vehicle loans, personal loans and loans for businesses and agriculture.",
    "Canara Bank provides education loan facilities to eligible students for pursuing higher education in India and abroad, subject to applicable eligibility conditions.",
    "Canara Bank provides digital banking services through internet banking and mobile banking platforms.",
    "Customers can use Canara Bank branches and digital banking channels for various banking transactions and services.",
    "Canara Bank provides financial services for individuals, businesses, farmers, students and other customer segments.",
    "Interest rates, processing fees, eligibility requirements and documentation requirements may vary depending on the banking product and applicable bank policies.",
    "Customers should contact Canara Bank through official channels or visit a Canara Bank branch for the latest information regarding banking products and services.",
    "Customers should never share their ATM PIN, UPI PIN, internet banking password, OTP, CVV or other confidential banking credentials with anyone.",
    "Canara Bank provides customer support through its official customer service channels for assistance with banking-related queries.",
    "Canara Bank offers various deposit products including savings deposits, current deposits and term deposits."
]

text_documents = [
    Document(page_content=text)
    for text in raw_paragraphs
]

urls = [
    "https://canarabank.com/","https://www.canarabank.bank.in/pages/loans-interest-rates"
]

web_documents = []

for url in urls:
    try:
        loader = WebBaseLoader(url)
        documents = loader.load()
        web_documents.extend(documents)
        print(f"Loaded: {url}")
    except Exception as e:
        print(f"Could not load {url}: {e}")

pdf_path = "CNRB_SB.pdf","CNRB_CB.pdf"

try:
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_documents = pdf_loader.load()
    print("PDF loaded successfully.")
except Exception:
    print("PDF not found. Continuing without PDF.")
    pdf_documents = []

all_documents = text_documents + web_documents + pdf_documents

print(f"Total documents loaded: {len(all_documents)}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(all_documents)

print(f"Total searchable chunks: {len(chunks)}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created.")

groq_api_key = userdata.get("GROQAPI")

if not groq_api_key:
    raise ValueError(
        "GROQAPI not found. Add your Groq API key to Colab Secrets."
    )

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

print("\nCanara Bank AI Assistant Online.")
print("Type 'exit' to stop.")
print("-" * 60)

while True:

    user_query = input("\nYou: ").strip()

    if user_query.lower() == "exit":
        print("Thank You! Visit Again...")
        break

    if not user_query:
        print("Please enter a question.")
        continue

    relevant_chunks = vector_db.similarity_search(
        user_query,
        k=4
    )

    retrieved_context = "\n\n".join(
        [doc.page_content for doc in relevant_chunks]
    )

    system_prompt = f"""
You are a helpful AI assistant for Canara Bank.

Answer the user's question using ONLY the information
provided in the context below.

Rules:

1. Do not guess or invent information.
2. Do not provide unsupported interest rates, fees,
   charges, eligibility criteria or limits.
3. If the information is not available in the context,
   say: "I do not have enough information to answer that."
4. For account-specific information, advise the user
   to contact Canara Bank through official channels.
5. Never ask the user for OTP, ATM PIN, UPI PIN,
   CVV, password or other confidential banking information.
6. Do not claim to be a Canara Bank employee.
7. For current banking information, advise the user
   to verify the information with Canara Bank.

CONTEXT:
{retrieved_context}
"""

    try:

        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_query
                }
            ],
            temperature=0
        )

        print(
            f"AI: {response.choices[0].message.content}"
        )

    except Exception as e:
        print(f"Error: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

/tmp/ipykernel_1917/2325373412.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader


Step 1: Ingesting Canara Bank data...
Loaded: https://canarabank.com/
Loaded: https://www.canarabank.bank.in/pages/loans-interest-rates
PDF not found. Continuing without PDF.
Total documents loaded: 14
Total searchable chunks: 194


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS vector database created.

Canara Bank AI Assistant Online.
Type 'exit' to stop.
------------------------------------------------------------
AI: I do not have enough information to answer that.
AI: The rate of interest for **Agricultural Gold Loan schemes** is:

**1 Year MCLR + 0.10% (effective from 12 August 2026).**
AI: The information available indicates that the **Swarna Loans** under the Swarna Schemes are priced at:

**Rate of Interest:** **RLLR + 0.95%** (effective from 12 August 2026)

No other specific interest‑rate details for the additional Swarna products (such as Swarna Overdraft, Swarna Express, or Swarna Monthly Interest) are provided in the given context. If you need further details, please verify with Canara Bank through its official channels.
AI: I do not have enough information to answer that.
AI: Canara Bank is one of India’s major public‑sector banks. It was founded in 1906 and is headquartered in Bengaluru, Karnataka.  

The bank offers a wide range of finan

KeyboardInterrupt: Interrupted by user